In [1]:
import sys
import os

# Make sure the notebook can find the source code in the src folder
sys.path.append(os.path.abspath('../'))
from src.data_loader import download_stock_data

# List of selected stocks
omxs30_tickers = [
    'INVE-B.ST', 'VOLV-B.ST', 'ATCO-A.ST', 'SEB-A.ST', 'SHB-A.ST',
    'SWED-A.ST', 'AZN.ST', 'ALFA.ST', 'ERIC-B.ST', 'SAND.ST',
    'HEXA-B.ST', 'HM-B.ST', 'ABB.ST', 'ASSA-B.ST', 'SKF-B.ST'
]

# Run the function and fetch the data
df_prices = download_stock_data(
    tickers=omxs30_tickers, 
    start_date='2015-01-01', 
    end_date='2026-01-01'
)

# Show the first 20 rows of the new price history
df_prices.head(20)

Fetching data for 15 assets from Yahoo Finance...


[*********************100%***********************]  15 of 15 completed

Ticker,ABB.ST,ALFA.ST,ASSA-B.ST,ATCO-A.ST,AZN.ST,ERIC-B.ST,HEXA-B.ST,HM-B.ST,INVE-B.ST,SAND.ST,SEB-A.ST,SHB-A.ST,SKF-B.ST,SWED-A.ST,VOLV-B.ST
Date,,,,,,,,,,,,,,,
2015-01-02,153.346970,116.762268,112.851395,31.207474,517.489746,67.695740,33.353165,203.669815,54.223530,54.751968,49.862499,51.403088,112.249344,85.793808,46.302238
2015-01-05,151.868881,114.624916,115.180183,31.150084,521.682068,67.910774,33.615032,202.735535,53.786411,54.176399,49.285034,50.940754,112.727303,85.264503,45.949623
2015-01-07,146.972870,111.300163,111.371910,30.274834,501.187347,67.229874,33.091305,198.375610,51.999855,52.521606,47.326694,49.833950,109.245132,83.058998,45.678379
2015-01-08,150.575592,114.466606,114.248695,31.322264,517.489746,70.132645,33.752861,203.358414,52.817112,53.492893,47.703289,50.408363,112.522469,82.573792,47.305866
2015-01-09,148.358551,111.300163,114.385666,31.293560,519.818909,69.810112,34.207676,202.237289,52.360954,52.881340,46.950085,49.455681,112.795563,81.382812,47.360123
2015-01-12,148.820435,109.716957,117.317207,31.566183,531.929504,70.598526,34.841660,202.610962,52.341961,53.097183,47.050507,49.875980,114.365967,81.603371,48.282368
2015-01-13,147.804291,109.241989,118.440506,31.580523,537.518616,69.774284,34.786526,206.472580,53.311260,53.744709,46.975189,50.842682,118.257835,82.265022,49.367374
2015-01-14,143.739655,107.579597,115.399384,30.662243,533.792358,69.451736,34.152534,202.860123,51.790798,51.262531,46.397732,49.679840,116.482613,80.456512,47.983994
2015-01-15,149.467072,109.796112,118.248734,31.164429,542.176636,69.666763,34.042286,205.725189,52.570038,52.197845,47.602863,50.254253,118.326096,80.765289,49.150364


In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os

# This line tells Python to look in the folder above this notebook
sys.path.append(os.path.abspath('../'))

from src.data_loader import download_stock_data, download_macro_and_market_data, get_data_path

# 1. Fetch stock data
omxs30_tickers = [
    'INVE-B.ST', 'VOLV-B.ST', 'ATCO-A.ST', 'SEB-A.ST', 'SHB-A.ST',
    'SWED-A.ST', 'AZN.ST', 'ALFA.ST', 'ERIC-B.ST', 'SAND.ST',
    'HEXA-B.ST', 'HM-B.ST', 'ABB.ST', 'ASSA-B.ST', 'SKF-B.ST'
]
df_prices = download_stock_data(tickers=omxs30_tickers, start_date='2015-01-01', end_date='2026-01-01')
df_monthly_returns = df_prices.resample('ME').last().pct_change().dropna()

# 2. Fetch all macro and market data
df_global_data = download_macro_and_market_data(start_date='2015-01-01', end_date='2026-01-01')
df_monthly_global = df_global_data.resample('ME').last()

# 3. Show the first rows of the macro data
df_monthly_global.head()

[**********************53%                       ]  8 of 15 completed

Fetching data for 15 assets from Yahoo Finance...


[*********************100%***********************]  15 of 15 completed

Fetching global market and macro data (VIX, rates, currencies, commodities, indices)...



[*********************100%***********************]  8 of 8 completed


Ticker,Brent_Oil,EURSEK,Copper,USDSEK,SP500,OMXS30_Index,US10Y,VIX
Date,,,,,,,,
2015-01-31,52.990002,9.3625,2.5280,8.26040,1994.989990,1573.619995,1.675,20.969999
2015-02-28,62.580002,9.4023,2.7160,8.39220,2104.500000,1691.030029,2.002,13.340000
2015-03-31,55.110001,9.3018,2.7470,8.58890,2067.889893,1667.729980,1.934,15.290000
2015-04-30,66.779999,9.2690,2.8865,8.34131,2085.510010,1628.040039,2.046,14.550000
2015-05-31,65.559998,9.2674,2.7595,8.45560,2107.389893,1644.989990,2.095,13.840000


In [3]:
# 1. Install the package required for Excel export directly in the cell
!pip3 install openpyxl

# 2. Merge stock returns and macro data on date
df_combined = df_monthly_returns.join(df_monthly_global, how='inner')

# 3. Export to Excel (the file will be saved in your project folder)
df_combined.to_excel(get_data_path('omxs30_och_makrodata.xlsx'))

print("Done! The file 'omxs30_och_makrodata.xlsx' has been created in data/.")


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Done! The file 'omxs30_och_makrodata.xlsx' has been created in data/.


In [4]:
import pandas as pd
import numpy as np
from src.data_loader import get_data_path

# 1. Create the shifted target variable as next month's relative return
# Relative return = stock return - OMXS30 return
omxs30_return_next = df_monthly_global['OMXS30_Index'].pct_change().shift(-1)
df_target = df_monthly_returns.shift(-1).sub(omxs30_return_next, axis=0)

# 2. Create stock-specific rolling features
df_momentum_3M = df_monthly_returns.rolling(3).sum()
df_volatility_3M = df_monthly_returns.rolling(3).std()
df_omxs30_momentum_3M = df_monthly_global['OMXS30_Index'].pct_change().rolling(3).sum()

# 3. Transform the stock target from wide to long format
df_target_long = df_target.reset_index().melt(
    id_vars='Date', 
    var_name='Ticker', 
    value_name='Target_Next_Month_Relative_Return'
)

# 4. Transform momentum 1M to 12M into long format and merge them
momentum_feature_frames = []
for months in range(1, 13):
    df_momentum_month = df_monthly_returns.rolling(months).sum()
    df_momentum_long = df_momentum_month.reset_index().melt(
        id_vars='Date', 
        var_name='Ticker', 
        value_name=f'Feature_Momentum_{months}M'
    )
    momentum_feature_frames.append(df_momentum_long)

df_stock_features_long = momentum_feature_frames[0]
for df_feature in momentum_feature_frames[1:]:
    df_stock_features_long = pd.merge(df_stock_features_long, df_feature, on=['Date', 'Ticker'])

# 5. Add 3M volatility
df_volatility_long = df_volatility_3M.reset_index().melt(
    id_vars='Date', 
    var_name='Ticker', 
    value_name='Feature_Volatility_3M'
 )

# 6. Add relative momentum and momentum x rate sensitivity
df_momentum_3M_long = df_momentum_3M.reset_index().melt(
    id_vars='Date', 
    var_name='Ticker', 
    value_name='Feature_Momentum_3M'
 )

df_omxs30_momentum_long = df_omxs30_momentum_3M.reset_index()
df_omxs30_momentum_long.columns = ['Date', 'OMXS30_Momentum_3M']

df_relative_momentum_long = pd.merge(df_momentum_3M_long, df_omxs30_momentum_long, on='Date', how='left')
df_relative_momentum_long['Feature_Relative_Momentum_3M'] = df_relative_momentum_long['Feature_Momentum_3M'] - df_relative_momentum_long['OMXS30_Momentum_3M']
df_relative_momentum_long = df_relative_momentum_long[['Date', 'Ticker', 'Feature_Relative_Momentum_3M']]

df_rate_sensitivity_long = pd.merge(
    df_momentum_3M_long[['Date', 'Ticker', 'Feature_Momentum_3M']],
    df_monthly_global.reset_index()[['Date', 'US10Y']],
    on='Date',
    how='left'
 )
df_rate_sensitivity_long['Feature_Mom3M_x_US10Y'] = df_rate_sensitivity_long['Feature_Momentum_3M'] * df_rate_sensitivity_long['US10Y']
df_rate_sensitivity_long = df_rate_sensitivity_long[['Date', 'Ticker', 'Feature_Mom3M_x_US10Y']]

# 7. Merge all stock-specific features together
df_stocks_long = pd.merge(df_target_long, df_stock_features_long, on=['Date', 'Ticker'])
df_stocks_long = pd.merge(df_stocks_long, df_volatility_long, on=['Date', 'Ticker'])
df_stocks_long = pd.merge(df_stocks_long, df_relative_momentum_long, on=['Date', 'Ticker'])
df_stocks_long = pd.merge(df_stocks_long, df_rate_sensitivity_long, on=['Date', 'Ticker'])

# 8. Merge with the macro data by date
df_macro_reset = df_monthly_global.reset_index()
df_panel = pd.merge(df_stocks_long, df_macro_reset, on='Date')

# 9. Remove rows with NaN values (at the start because of rolling windows, at the end because of the shift)
df_panel = df_panel.dropna().reset_index(drop=True)

# 10. Export the new structure to Excel
df_panel.to_excel(get_data_path('omxs30_paneldata_slutgiltig.xlsx'), index=False)

print(f"Done! The panel data has been created with a total of {df_panel.shape[0]} rows.")
print("Open data/omxs30_paneldata_slutgiltig.xlsx to review the result.")

Done! The panel data has been created with a total of 1785 rows.
Open data/omxs30_paneldata_slutgiltig.xlsx to review the result.


In [5]:
from sklearn.preprocessing import StandardScaler

# 1. Ensure the date column has the correct format
df_panel['Date'] = pd.to_datetime(df_panel['Date'])

# 2. Create a time split (train through 2022, test from 2023 onward)
train_mask = df_panel['Date'] < '2023-01-01'
test_mask = df_panel['Date'] >= '2023-01-01'

df_train = df_panel[train_mask].copy()
df_test = df_panel[test_mask].copy()

# 3. Define which columns are features (X) and which one is the target (y)
feature_cols = [
    'Feature_Momentum_1M',
    'Feature_Momentum_2M',
    'Feature_Momentum_3M',
    'Feature_Momentum_4M',
    'Feature_Momentum_5M',
    'Feature_Momentum_6M',
    'Feature_Momentum_7M',
    'Feature_Momentum_8M',
    'Feature_Momentum_9M',
    'Feature_Momentum_10M',
    'Feature_Momentum_11M',
    'Feature_Momentum_12M',
    'Feature_Volatility_3M',
    'Feature_Relative_Momentum_3M',
    'Feature_Mom3M_x_US10Y',
    'VIX',
    'US10Y',
    'USDSEK', 
    'EURSEK',
    'Copper',
    'Brent_Oil',
    'SP500',
    'OMXS30_Index'
 ]
target_col = 'Target_Next_Month_Relative_Return'

# 4. Initialize the scaling tool (StandardScaler shifts to mean 0 and standard deviation 1)
scaler = StandardScaler()

# 5. Fit the tool on the training data and transform it
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])

# 6. Transform the test data with exactly the same rules (use only .transform, not .fit_transform)
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

print(f"The data has been split and normalized!")
print(f"Training data (2015-2022): {df_train.shape[0]} rows")
print(f"Test data (2023-2026): {df_test.shape[0]} rows")

# Show what the scaled training data looks like now
df_train.head()

The data has been split and normalized!
Training data (2015-2022): 1260 rows
Test data (2023-2026): 525 rows


,Date,Ticker,Target_Next_Month_Relative_Return,Feature_Momentum_1M,Feature_Momentum_2M,Feature_Momentum_3M,Feature_Momentum_4M,Feature_Momentum_5M,Feature_Momentum_6M,Feature_Momentum_7M,...,Feature_Relative_Momentum_3M,Feature_Mom3M_x_US10Y,Brent_Oil,EURSEK,Copper,USDSEK,SP500,OMXS30_Index,US10Y,VIX
0,2016-01-31,ABB.ST,0.036275,-0.643439,-1.260261,-0.974584,-0.295774,-1.002221,-1.427463,-1.335748,...,-0.014117,-0.852929,-1.491438,-1.874131,-1.427630,-0.754193,-1.513804,-1.343626,-0.155818,0.106587
1,2016-02-29,ABB.ST,0.032203,0.522587,-0.086507,-0.732913,-0.569022,0.010202,-0.662498,-1.088185,...,0.436429,-0.597091,-1.429839,-1.766464,-1.339094,-0.674510,-1.524106,-1.286731,-0.394225,0.150954
2,2016-03-31,ABB.ST,0.074317,0.233916,0.569661,0.082059,-0.517050,-0.390042,0.129866,-0.501654,...,0.885623,0.060771,-1.248045,-2.046556,-1.266343,-1.284740,-1.360110,-1.310724,-0.336807,-0.685686
3,2016-04-30,ABB.ST,0.014179,0.830688,0.800087,1.020086,0.553851,-0.031423,0.047932,0.507730,...,1.427590,0.848866,-0.820855,-2.164518,-1.132508,-1.380708,-1.352959,-1.328227,-0.295617,-0.463850
4,2016-05-31,ABB.ST,-0.008631,0.143320,0.732330,0.778748,1.002604,0.589038,0.048182,0.122687,...,1.180244,0.654554,-0.742729,-1.916699,-1.381646,-1.014322,-1.312253,-1.297465,-0.276894,-0.655263


In [6]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# 1. Separate features (X) and target (y) for both training and test
X_train = df_train[feature_cols]
y_train = df_train[target_col]

X_test = df_test[feature_cols]
y_test = df_test[target_col]

# 2. Create and train the model (alpha is the regularization strength)
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# 3. Make predictions on both the training and test data
df_train['Predicted_Return'] = model.predict(X_train)
df_test['Predicted_Return'] = model.predict(X_test)

# 4. Evaluate the model with basic metrics
print("--- TRAINING PERFORMANCE Ridge (In-Sample) ---")
print(f"R2-score: {r2_score(y_train, df_train['Predicted_Return']):.4f}")
print(f"MSE: {mean_squared_error(y_train, df_train['Predicted_Return']):.6f}")

print("\n--- TEST PERFORMANCE Ridge (Out-of-Sample) ---")
print(f"R2-score: {r2_score(y_test, df_test['Predicted_Return']):.4f}")
print(f"MSE: {mean_squared_error(y_test, df_test['Predicted_Return']):.6f}")

# 5. Save the test results to Excel so you can compare predictions against the ground truth
df_test[['Date', 'Ticker', 'Target_Next_Month_Relative_Return', 'Predicted_Return']].to_excel(get_data_path('omxs30_modell_gissningar.xlsx'), index=False)
print("\nDone! The predictions have been saved to 'omxs30_modell_gissningar.xlsx'.")

--- TRAINING PERFORMANCE Ridge (In-Sample) ---
R2-score: 0.0161
MSE: 0.003044

--- TEST PERFORMANCE Ridge (Out-of-Sample) ---
R2-score: -0.0415
MSE: 0.003016

Done! The predictions have been saved to 'omxs30_modell_gissningar.xlsx'.


In [7]:
from sklearn.ensemble import RandomForestRegressor

# 1. Create and configure the model (we add limits to reduce overfitting)
rf_model = RandomForestRegressor(
    n_estimators=100,    # Number of decision trees
    max_depth=4,         # Limit depth so the trees do not memorize noise
    random_state=42
)

# 2. Train the model
rf_model.fit(X_train, y_train)

# 3. Make predictions
df_train['RF_Predicted_Return'] = rf_model.predict(X_train)
df_test['RF_Predicted_Return'] = rf_model.predict(X_test)

# 4. Evaluate
print("--- RANDOM FOREST: TRAINING PERFORMANCE ---")
print(f"R2-score: {r2_score(y_train, df_train['RF_Predicted_Return']):.4f}")

print("\n--- RANDOM FOREST: TEST PERFORMANCE ---")
print(f"R2-score: {r2_score(y_test, df_test['RF_Predicted_Return']):.4f}")

--- RANDOM FOREST: TRAINING PERFORMANCE ---
R2-score: 0.1250

--- RANDOM FOREST: TEST PERFORMANCE ---
R2-score: 0.0299


In [8]:

# 1. Hämta betydelsen av varje feature från din tränade modell
importances = rf_model.feature_importances_
feature_names = X_train.columns

# 2. Skapa en tabell och sortera med den viktigaste överst
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 3. Skriv ut resultatet
print("--- MODELLENS VIKTIGASTE FEATURES ---")
print(importance_df.to_string(index=False))

--- MODELLENS VIKTIGASTE FEATURES ---
                     Feature  Importance
Feature_Relative_Momentum_3M    0.106042
        Feature_Momentum_10M    0.104491
         Feature_Momentum_4M    0.083467
        Feature_Momentum_12M    0.070885
        Feature_Momentum_11M    0.066023
         Feature_Momentum_6M    0.063793
                         VIX    0.061791
         Feature_Momentum_8M    0.053594
         Feature_Momentum_1M    0.052781
         Feature_Momentum_2M    0.043777
         Feature_Momentum_9M    0.040126
         Feature_Momentum_5M    0.035903
       Feature_Mom3M_x_US10Y    0.032990
         Feature_Momentum_3M    0.032597
       Feature_Volatility_3M    0.031214
         Feature_Momentum_7M    0.030372
                      Copper    0.017013
                       SP500    0.016442
                OMXS30_Index    0.015900
                       US10Y    0.012457
                      EURSEK    0.010069
                   Brent_Oil    0.009340
                   